# 📈 FinBERT + LSTM — Kaggle GPU Version

**Datasets required (add via "+ Add Data"):**
| Dataset | Kaggle path used |
|---|---|
| `ramavaibhav/nasdaq-external-data-csv` | `/kaggle/input/datasets/ramavaibhav/nasdaq-external-data-csv/` |
| `ramavaibhav/full-history` (price CSVs) | `/kaggle/input/full-history/full_history/` |

**Price CSV format:** `date, open, high, low, close, adj close, volume`  
**Outputs saved to:** `/kaggle/working/output/`

> Checkpoint system active — if the kernel times out, re-run and it resumes from last saved chunk.


In [1]:
import torch, gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM          : {mem:.1f} GB")
    SUGGESTED_BATCH = 256 if mem >= 16 else 128 if mem >= 8 else 64 if mem >= 4 else 32
else:
    SUGGESTED_BATCH = 16
    print("⚠️  No GPU — CPU mode (slow but works)")

print(f"Auto batch size: {SUGGESTED_BATCH}")


GPU available : True
GPU           : Tesla P100-PCIE-16GB
VRAM          : 17.1 GB
Auto batch size: 256


In [2]:
import os, gc, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")
print("✅ Imports OK")


✅ Imports OK


In [3]:
# ─────────────────────────────────────────────
# STEP 1: Config  (Kaggle paths)
# ─────────────────────────────────────────────

FILE_PATH    = "/kaggle/input/datasets/ramavaibhav/nasdaq-external-data-csv/nasdaq_exteral_data.csv"
PRICE_FOLDER = "/kaggle/input/datasets/ramavaibhav/full-history/full_history"
OUTPUT_DIR   = "/kaggle/working/output"
CKPT_DIR     = "/kaggle/working/output/sent_chunks"

OUTPUT_SENT  = "/kaggle/working/output/sentiment_output.csv"
OUTPUT_FINAL = "/kaggle/working/output/final_merged.csv"
MODEL_SAVE   = "/kaggle/working/output/best_lstm.pt"
PRED_SAVE    = "/kaggle/working/output/predictions.csv"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# ── Hyperparams ──────────────────────────────
CHUNKSIZE  = 50_000
BATCH_SIZE = SUGGESTED_BATCH   # auto-set by GPU cell above
MAX_LENGTH = 64                # tokens per text (fast + accurate)
SEQ_LEN    = 60
EPOCHS     = 30
LR         = 0.001
MAX_ROWS   = None              # set e.g. 200_000 for a quick smoke-test

USE_FP16   = torch.cuda.is_available()
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Sanity checks ────────────────────────────
assert os.path.isfile(FILE_PATH),   f"❌ News CSV not found: {FILE_PATH}"
assert os.path.isdir(PRICE_FOLDER), f"❌ Price folder not found: {PRICE_FOLDER}"

n_price_files = len([f for f in os.listdir(PRICE_FOLDER) if f.endswith(".csv")])

print("✅ Config ready")
print(f"   Device      : {device}  (FP16={USE_FP16})")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   News CSV    : {FILE_PATH}")
print(f"   Price files : {n_price_files} tickers in {PRICE_FOLDER}")
print(f"   Output dir  : {OUTPUT_DIR}")


✅ Config ready
   Device      : cuda  (FP16=True)
   Batch size  : 256
   News CSV    : /kaggle/input/datasets/ramavaibhav/nasdaq-external-data-csv/nasdaq_exteral_data.csv
   Price files : 7693 tickers in /kaggle/input/datasets/ramavaibhav/full-history/full_history
   Output dir  : /kaggle/working/output


In [4]:
# ─────────────────────────────────────────────
# STEP 2: Load FinBERT
# ─────────────────────────────────────────────
print("[1/7] Loading FinBERT (ProsusAI/finbert)...")
print("      First run downloads ~440 MB and caches it locally.")

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert   = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
finbert.to(device)
finbert.eval()

if USE_FP16:
    finbert = finbert.half()
    print("✅ FinBERT loaded  (FP16 — faster GPU inference)")
else:
    print("✅ FinBERT loaded  (FP32 — CPU mode)")


[1/7] Loading FinBERT (ProsusAI/finbert)...
      First run downloads ~440 MB and caches it locally.


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

✅ FinBERT loaded  (FP16 — faster GPU inference)


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ramavaibhav
/kaggle/input/datasets/ramavaibhav/full-history
/kaggle/input/datasets/ramavaibhav/full-history/full_history
/kaggle/input/datasets/ramavaibhav/nasdaq-external-data-csv


In [6]:
import os

price_folder = "/kaggle/input/datasets/ramavaibhav/full-history/full_history"

print(len(os.listdir(price_folder)))
print(os.listdir(price_folder)[:10])

7693
['MTL.csv', 'JOE.csv', 'DRE.csv', 'CTY.csv', 'AMSF.csv', 'IMH.csv', 'HAUZ.csv', 'GFL.csv', 'HDIV.csv', 'CLM.csv']


In [7]:
import os

print(os.listdir("/kaggle/input/datasets/ramavaibhav/full-history"))

['full_history']


In [8]:
# ─────────────────────────────────────────────
# STEP 3: FinBERT Sentiment Extraction
#         Checkpointed — safe to restart
# ─────────────────────────────────────────────

def get_finbert_sentiment(texts):
    clean = [str(t)[:512] if pd.notna(t) and str(t).strip() else "neutral" for t in texts]
    all_probs = []
    for i in range(0, len(clean), BATCH_SIZE):
        batch = clean[i : i + BATCH_SIZE]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=USE_FP16):
            logits = finbert(**enc).logits
        all_probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(all_probs)


# ── Resume from last checkpoint if any ──
done = sorted(
    int(f.replace("chunk_","").replace(".csv",""))
    for f in os.listdir(CKPT_DIR) if f.startswith("chunk_") and f.endswith(".csv")
)
resume_from = max(done) + 1 if done else 0
print(f"[2/7] Sentiment extraction  (resuming from chunk {resume_from})" if resume_from
      else "[2/7] Sentiment extraction  (fresh start)")

try:
    total_rows   = sum(1 for _ in open(FILE_PATH, encoding="utf-8", errors="ignore")) - 1
    if MAX_ROWS: total_rows = min(total_rows, MAX_ROWS)
    total_chunks = (total_rows // CHUNKSIZE) + 1
    remaining    = max(0, total_chunks - resume_from)
    print(f"      ~{total_rows:,} rows  →  {total_chunks} chunks  ({remaining} remaining)")
except:
    remaining = None

rows_read  = resume_from * CHUNKSIZE
chunk_idx  = 0
new_chunks = 0
t0         = time.time()

with tqdm(total=remaining, desc="📰 Sentiment", unit="chunk",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} chunks  [{elapsed}<{remaining}, {rate_fmt}]") as pbar:

    for chunk in pd.read_csv(FILE_PATH, chunksize=CHUNKSIZE,
                              usecols=["Date","Stock_symbol","Article_title","Article"],
                              on_bad_lines="skip", low_memory=False,
                              encoding="utf-8", encoding_errors="replace"):

        if chunk_idx < resume_from:
            chunk_idx += 1
            continue
        if MAX_ROWS and rows_read >= MAX_ROWS:
            break

        rows_read += len(chunk)

        texts = (chunk["Article_title"].fillna("") + ". " +
                 chunk["Article"].fillna("").str[:300]).tolist()
        probs = get_finbert_sentiment(texts)

        out = chunk[["Date","Stock_symbol"]].copy()
        out["pos"], out["neg"], out["neu"] = probs[:,0], probs[:,1], probs[:,2]
        out.to_csv(os.path.join(CKPT_DIR, f"chunk_{chunk_idx}.csv"), index=False)

        chunk_idx  += 1
        new_chunks += 1
        elapsed     = time.time() - t0
        speed       = (new_chunks * CHUNKSIZE) / elapsed if elapsed else 0
        pbar.set_postfix(rows=f"{rows_read:,}", speed=f"{speed:,.0f} r/s")
        pbar.update(1)

        if new_chunks % 20 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

# ── Combine all chunks ──
print("\n🔗 Combining chunks...")
parts = sorted(os.path.join(CKPT_DIR, f)
               for f in os.listdir(CKPT_DIR) if f.startswith("chunk_"))
sentiment_df = pd.concat(
    (pd.read_csv(p) for p in tqdm(parts, desc="📂 Loading", unit="file")),
    ignore_index=True
)
sentiment_df["Date"] = pd.to_datetime(sentiment_df["Date"], errors="coerce")
sentiment_df.dropna(subset=["Date","Stock_symbol"], inplace=True)
sentiment_df.to_csv(OUTPUT_SENT, index=False)
print(f"✅ Saved → {OUTPUT_SENT}  |  shape: {sentiment_df.shape}  |  time: {(time.time()-t0)/60:.1f} min")

del finbert
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()


[2/7] Sentiment extraction  (fresh start)
      ~94,639,684 rows  →  1893 chunks  (1893 remaining)


📰 Sentiment:   0%|          | 0/1893 chunks  [00:00<?, ?chunk/s]


🔗 Combining chunks...


📂 Loading:   0%|          | 0/311 [00:00<?, ?file/s]

✅ Saved → /kaggle/working/output/sentiment_output.csv  |  shape: (5744672, 5)  |  time: 362.3 min


19

In [9]:
# ─────────────────────────────────────────────
# STEP 4: Daily Sentiment Aggregation
# ─────────────────────────────────────────────
print("[3/7] Aggregating daily sentiment...")

daily_sentiment = (
    sentiment_df
    .groupby(["Stock_symbol","Date"])[["pos","neg","neu"]]
    .mean().reset_index()
)
daily_sentiment["sentiment_score"] = daily_sentiment["pos"] - daily_sentiment["neg"]

print(f"✅ Shape: {daily_sentiment.shape}")
print(daily_sentiment.head(3))


[3/7] Aggregating daily sentiment...
✅ Shape: (2857167, 6)
  Stock_symbol                      Date       pos       neg       neu  \
0            A 2009-04-29 00:00:00+00:00  0.067178  0.298872  0.633950   
1            A 2009-06-01 00:00:00+00:00  0.317068  0.045091  0.637842   
2            A 2009-07-14 00:00:00+00:00  0.202286  0.017525  0.780189   

   sentiment_score  
0        -0.231695  
1         0.271977  
2         0.184761  


In [10]:
# ─────────────────────────────────────────────
# STEP 5: Load Price Data
# Detected columns: date,open,high,low,close,adj close,volume
# ─────────────────────────────────────────────
print(f"[4/7] Loading prices from: {PRICE_FOLDER}")

csv_files = [f for f in os.listdir(PRICE_FOLDER) if f.endswith(".csv")]
print(f"      {len(csv_files)} ticker files found")

price_chunks = []
skipped      = []

for fname in tqdm(csv_files, desc="💹 Prices", unit="ticker",
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]"):
    fpath  = os.path.join(PRICE_FOLDER, fname)
    ticker = os.path.splitext(fname)[0].upper()
    try:
        df = pd.read_csv(fpath, low_memory=False)

        # ── Normalise column names (lowercase, strip spaces) ──
        df.columns = df.columns.str.strip().str.lower()

        # ── Rename to standard names ──
        df.rename(columns={
            "date"      : "Date",
            "open"      : "Open",
            "high"      : "High",
            "low"       : "Low",
            "close"     : "Close",
            "adj close" : "Adj_Close",   # keep separately
            "volume"    : "Volume"
        }, inplace=True)

        # Use Adj_Close as Close if present (more accurate for older data)
        if "Adj_Close" in df.columns:
            df["Close"] = df["Adj_Close"]

        df["Stock_symbol"] = ticker

        keep = [c for c in ["Date","Open","High","Low","Close","Volume","Stock_symbol"]
                if c in df.columns]
        price_chunks.append(df[keep])

    except Exception as e:
        skipped.append(f"{fname}: {e}")

if skipped:
    print(f"  ⚠️  Skipped {len(skipped)} files")

prices = pd.concat(price_chunks, ignore_index=True)
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
prices.dropna(subset=["Date","Close"], inplace=True)
prices["Close"] = pd.to_numeric(prices["Close"], errors="coerce")
prices.dropna(subset=["Close"], inplace=True)
prices.sort_values(["Stock_symbol","Date"], inplace=True)

print(f"✅ Prices shape  : {prices.shape}")
print(f"   Date range    : {prices['Date'].min().date()} → {prices['Date'].max().date()}")
print(f"   Unique tickers: {prices['Stock_symbol'].nunique()}")
print(prices.head(3))


[4/7] Loading prices from: /kaggle/input/datasets/ramavaibhav/full-history/full_history
      7693 ticker files found


💹 Prices:   0%|          | 0/7693 [00:00<?]

✅ Prices shape  : (29677465, 7)
   Date range    : 1962-01-02 → 2023-12-28
   Unique tickers: 7693
               Date       Open       High        Low      Close      Volume  \
27789890 1999-11-18  32.546494  35.765381  28.612303  27.068665  62546300.0   
27789889 1999-11-19  30.713520  30.758226  28.478184  24.838577  15234100.0   
27789888 1999-11-22  29.551144  31.473534  28.657009  27.068665   6577800.0   

         Stock_symbol  
27789890            A  
27789889            A  
27789888            A  


In [11]:
# ─────────────────────────────────────────────
# STEP 6: Merge Price + Sentiment
# ─────────────────────────────────────────────
print("[5/7] Merging...")

# Ensure same datetime format
prices["Date"] = pd.to_datetime(prices["Date"])
daily_sentiment["Date"] = pd.to_datetime(daily_sentiment["Date"]).dt.tz_localize(None)

data = pd.merge(
    prices,
    daily_sentiment,
    on=["Stock_symbol","Date"],
    how="inner"
)

data.sort_values(["Stock_symbol","Date"], inplace=True)

data.to_csv(OUTPUT_FINAL, index=False)

print(f"✅ Merged shape: {data.shape}")
print(f"Tickers with both news + price: {data['Stock_symbol'].nunique()}")
print(f"Saved → {OUTPUT_FINAL}")

print(data.head(3))


[5/7] Merging...
✅ Merged shape: (2093589, 11)
Tickers with both news + price: 6008
Saved → /kaggle/working/output/final_merged.csv
        Date       Open       High        Low      Close     Volume  \
0 2009-04-29  12.310444  13.011445  12.238913  11.772207  5516500.0   
1 2009-06-01  13.190271  13.741058  13.190271  12.327197  5764500.0   
2 2009-07-14  13.769670  14.020029  13.769670  12.725476  2331800.0   

  Stock_symbol       pos       neg       neu  sentiment_score  
0            A  0.067178  0.298872  0.633950        -0.231695  
1            A  0.317068  0.045091  0.637842         0.271977  
2            A  0.202286  0.017525  0.780189         0.184761  


In [12]:
# ─────────────────────────────────────────────
# STEP 7: Build t-60 Sequences
# ─────────────────────────────────────────────
print(f"[6/7] Building sequences (SEQ_LEN={SEQ_LEN})...")

FEATURE_COLS = ["Close","Volume","pos","neg","neu","sentiment_score"]
X_list, y_list = [], []
scaler_map     = {}
skipped_tickers = []

for ticker in tqdm(data["Stock_symbol"].unique(), desc="🧱 Sequences", unit="ticker",
                   bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]"):
    grp = (data[data["Stock_symbol"] == ticker]
           .dropna(subset=FEATURE_COLS)
           .sort_values("Date")
           .reset_index(drop=True))

    if len(grp) < SEQ_LEN + 1:
        skipped_tickers.append(ticker)
        continue

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(grp[FEATURE_COLS].values.astype(float))
    scaler_map[ticker] = scaler

    for i in range(SEQ_LEN, len(scaled)):
        X_list.append(scaled[i - SEQ_LEN : i])
        y_list.append(scaled[i, 0])   # next-day Close

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

print(f"✅ X: {X.shape}  |  y: {y.shape}")
print(f"   Tickers used   : {len(scaler_map)}")
print(f"   Tickers skipped: {len(skipped_tickers)}  (< {SEQ_LEN+1} data points)")

n       = len(X)
n_train = int(n * 0.70)
n_val   = int(n * 0.85)

X_train, y_train = X[:n_train],       y[:n_train]
X_val,   y_val   = X[n_train:n_val],  y[n_train:n_val]
X_test,  y_test  = X[n_val:],         y[n_val:]

print(f"   Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")

def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32).to(device)

train_loader = DataLoader(TensorDataset(to_tensor(X_train), to_tensor(y_train)),
                          batch_size=256, shuffle=True, num_workers=0)
val_loader   = DataLoader(TensorDataset(to_tensor(X_val),   to_tensor(y_val)),
                          batch_size=256, num_workers=0)
print("✅ DataLoaders ready")


[6/7] Building sequences (SEQ_LEN=60)...


🧱 Sequences:   0%|          | 0/6008 [00:00<?]

✅ X: (1788338, 60, 6)  |  y: (1788338,)
   Tickers used   : 4399
   Tickers skipped: 1609  (< 61 data points)
   Train: 1,251,836  |  Val: 268,251  |  Test: 268,251
✅ DataLoaders ready


In [13]:
# ─────────────────────────────────────────────
# STEP 8: LSTM Model
# ─────────────────────────────────────────────
class LSTMPredictor(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :])).squeeze(1)

lstm_model = LSTMPredictor(X.shape[2]).to(device)
n_params   = sum(p.numel() for p in lstm_model.parameters())
print(f"✅ LSTM ready  |  input_size={X.shape[2]}  |  params={n_params:,}  |  device={device}")


✅ LSTM ready  |  input_size=6  |  params=201,857  |  device=cuda


In [14]:
# ─────────────────────────────────────────────
# STEP 9: Training
# ─────────────────────────────────────────────
print("[7/7] Training LSTM...")

optimizer  = torch.optim.Adam(lstm_model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
                 optimizer, mode="min", patience=3, factor=0.5)
loss_fn    = nn.MSELoss()
scaler_amp = torch.cuda.amp.GradScaler(enabled=USE_FP16)

best_val   = float("inf")
patience_c = 0
EARLY_STOP = 7
history    = []

epoch_bar = tqdm(range(1, EPOCHS + 1), desc="🏋️  Epochs",
                 bar_format="{l_bar}{bar}| epoch {n_fmt}/{total_fmt}  [{elapsed}<{remaining}]")

for epoch in epoch_bar:
    # ── Train ──
    lstm_model.train()
    train_loss = 0.0
    for xb, yb in tqdm(train_loader, desc=f"  ep{epoch:02d} train", leave=False,
                        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} batches [{elapsed}<{remaining}]"):
        with torch.cuda.amp.autocast(enabled=USE_FP16):
            loss = loss_fn(lstm_model(xb), yb)
        optimizer.zero_grad()
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_loader.dataset)

    # ── Validate ──
    lstm_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                val_loss += loss_fn(lstm_model(xb), yb).item() * len(xb)
    val_loss /= len(val_loader.dataset)

    scheduler.step(val_loss)
    lr_now = optimizer.param_groups[0]["lr"]
    history.append({"epoch": epoch, "train_mse": train_loss, "val_mse": val_loss, "lr": lr_now})
    epoch_bar.set_postfix(train=f"{train_loss:.5f}", val=f"{val_loss:.5f}", lr=f"{lr_now:.5f}")

    if val_loss < best_val:
        best_val, patience_c = val_loss, 0
        torch.save(lstm_model.state_dict(), MODEL_SAVE)
    else:
        patience_c += 1
        if patience_c >= EARLY_STOP:
            print(f"\n⏹  Early stop at epoch {epoch}  (patience={EARLY_STOP})")
            break

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_log.csv"), index=False)
print(f"\n✅ Done  |  Best Val MSE: {best_val:.6f}  →  {MODEL_SAVE}")


[7/7] Training LSTM...


🏋️  Epochs:   0%|          | epoch 0/30  [00:00<?]

  ep01 train:   0%|          | 0/4890 batches [00:00<?]

  ep02 train:   0%|          | 0/4890 batches [00:00<?]

  ep03 train:   0%|          | 0/4890 batches [00:00<?]

...

  ep27 train:   0%|          | 0/4890 batches [00:00<?]

  ep28 train:   0%|          | 0/4890 batches [00:00<?]

  ep29 train:   0%|          | 0/4890 batches [00:00<?]

  ep30 train:   0%|          | 0/4890 batches [00:00<?]


✅ Done  |  Best Val MSE: 0.001222  →  /kaggle/working/output/best_lstm.pt


In [15]:
# ─────────────────────────────────────────────
# STEP 10: Test Evaluation
# ─────────────────────────────────────────────
lstm_model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
lstm_model.eval()

with torch.no_grad():
    preds = lstm_model(to_tensor(X_test)).cpu().numpy()

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n{'='*45}")
print(f"  Test MAE  (scaled 0–1) : {mae:.6f}")
print(f"  Test RMSE (scaled 0–1) : {rmse:.6f}")
print(f"{'='*45}")
print("  Values are in MinMax-scaled space [0,1] per ticker.")
print("  Lower = better. A well-trained model typically hits < 0.02 RMSE.")


OutOfMemoryError: CUDA out of memory. Tried to allocate 163.60 GiB. GPU 0 has a total capacity of 15.89 GiB of which 4.08 GiB is free. Process 3281 has 11.81 GiB memory in use. Of the allocated memory 11.49 GiB is allocated by PyTorch, and 22.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ─────────────────────────────────────────────
# STEP 11: Save All Outputs
# ─────────────────────────────────────────────
pd.DataFrame({"actual": y_test, "predicted": preds}).to_csv(PRED_SAVE, index=False)

print("🎉 Pipeline complete!")
print()
print(f"  📄 Predictions    → {PRED_SAVE}")
print(f"  🧠 Best model     → {MODEL_SAVE}")
print(f"  📊 Training log   → {os.path.join(OUTPUT_DIR, 'training_log.csv')}")
print(f"  🔗 Merged data    → {OUTPUT_FINAL}")
print(f"  💬 Sentiments     → {OUTPUT_SENT}")
